# Test the Cursor API key with LangChain

The Cursor API (`https://api.cursor.com`) is **not a chat model**. It launches and manages Cursor Cloud Agents, so it can't be used as `llm=...` in a chain.

The LangChain way to use it is to wrap its endpoints as **tools** and let an LLM (Groq here) decide when to call them.

1. **Test 1**: call the tools directly. This alone proves the key works.
2. **Test 2**: give the tools to a LangChain agent and ask a question in plain English.

All calls are read-only (`GET`). Nothing is created in your Cursor account.

In [2]:
import os
import requests
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

cursor_api_key = os.environ["CURSOR_API_KEY"]
groq_api_key = os.environ["GROQ_API_KEY"]

CURSOR_BASE_URL = "https://api.cursor.com"

## Wrap the Cursor API as LangChain tools

The `@tool` decorator turns a function into a LangChain tool. The **docstring** becomes the tool description the LLM reads to decide when to use it.

In [3]:
from langchain_core.tools import tool


def cursor_get(path: str) -> dict:
    """GET a Cursor API endpoint. Auth = API key as username, empty password."""
    resp = requests.get(f"{CURSOR_BASE_URL}{path}", auth=(cursor_api_key, ""), timeout=15)
    if resp.status_code == 401:
        return {"error": "Cursor API key is INVALID (401 Unauthorized)"}
    resp.raise_for_status()
    return resp.json()


@tool
def get_cursor_account() -> dict:
    """Check the Cursor API key and return who it belongs to (key name, user name, created date)."""
    return cursor_get("/v1/me")


@tool
def list_cursor_models() -> list:
    """List the model IDs this Cursor API key can use for Cloud Agents."""
    data = cursor_get("/v1/models")
    if "error" in data:
        return [data]
    return [m["id"] for m in data.get("items", [])]


@tool
def list_cursor_agents(limit: int = 5) -> list:
    """List the most recent Cursor Cloud Agents for this API key, newest first."""
    data = cursor_get(f"/v1/agents?limit={limit}")
    if "error" in data:
        return [data]
    return data.get("items", [])


tools = [get_cursor_account, list_cursor_models, list_cursor_agents]

## Test 1: call the tools directly

If this prints your key name, the key works. If it prints `INVALID`, check `CURSOR_API_KEY` in `.env`.

In [4]:
account = get_cursor_account.invoke({})
print("Account:", account)

models = list_cursor_models.invoke({})
print(f"\n{len(models)} models available:")
for m in models:
    print(" -", m)

Account: {'apiKeyName': 'cursor_api_key', 'userId': 348983737, 'createdAt': '2026-11-09T00:23:48.596Z', 'userEmail': 'mdkashifraza.ansari@sureify.com', 'userFirstName': 'Md Kashif Raza', 'userLastName': 'Ansari'}

48 models available:
 - auto-smart
 - default
 - grok-4.7
 - grok-4.6
 - composer-2.5
 - claude-opus-5-5
 - claude-opus-5
 - claude-opus-4-8
 - gemini-4-argon
 - muse-wonder-1.0
 - gpt-5.6-sol
 - gpt-5.5
 - claude-fable-5-1
 - claude-fable-5
 - grok-4.5
 - gemini-3.8-flash
 - gemini-3.7-flash
 - muse-spark-1.3
 - gpt-5.6-terra
 - claude-sonnet-5-5
 - claude-sonnet-5
 - claude-sonnet-4-6
 - gpt-5.3-codex
 - claude-opus-4-7
 - gpt-5.4
 - claude-opus-4-6
 - claude-opus-4-5
 - gpt-5.2
 - gpt-5.6-luna
 - gemini-3.6-flash
 - gemini-3.1-pro
 - gpt-5.4-mini
 - gpt-5.4-nano
 - claude-haiku-5-5
 - claude-haiku-4-5
 - claude-sonnet-4-5
 - gpt-5.1
 - gemini-3-flash
 - gemini-3.5-flash
 - claude-sonnet-4
 - gpt-5-mini
 - gemini-2.5-flash
 - kimi-k3
 - kimi-k2.7-code
 - glm-5.2
 - glm-5p3


## Test 2: let a LangChain agent use the tools

`create_agent` gives the LLM the tools. The LLM reads your question, picks which tools to call, reads their results, and writes the answer.

In [5]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=groq_api_key,
    reasoning_effort="low",  # less hidden 'thinking' -> more tokens left for the answer
    max_tokens=2000,
    timeout=30,
    max_retries=3,
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You help the user check their Cursor API key. Always use the tools to answer, never guess. "
        "Keep the answer short: say whether the key works, who owns it, how many models are available, "
        "and list at most 10 model IDs."
    ),
)

In [6]:
question = "Is my Cursor API key working? Who does it belong to, and which models can I use?"

result = agent.invoke({"messages": [{"role": "user", "content": question}]})

# Which tools did the LLM decide to call?
for msg in result["messages"]:
    for call in getattr(msg, "tool_calls", None) or []:
        print("Tool called:", call["name"], call["args"])

print("\nAnswer:\n")
print(result["messages"][-1].content)

Tool called: get_cursor_account {}
Tool called: list_cursor_models {}

Answer:

✅ Your Cursor API key is working.  
- **Owner:** Md Kashif Raza (mdkashifraza.ansari@sureify.com)  
- **Models available:** 50+ (list truncated to 10)  

**First 10 model IDs you can use:**  
1. auto-smart  
2. default  
3. grok-4.7  
4. grok-4.6  
5. composer-2.5  
6. claude‑opus‑5‑5  
7. claude‑opus‑5  
8. claude‑opus‑4‑8  
9. gemini‑4‑argon  
10. muse‑wonder‑1.0
